# SlimTSF Leakage-Free Evaluation
This notebook builds a leakage-aware evaluation using event-grouped splits, TSS/HSS threshold selection, and a temporal holdout. It keeps the feature set univariate (p3_flux_ic) for now.

In [1]:
import json
import math
import re
from pathlib import Path

import importlib
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import GroupKFold

import slimTSF
importlib.reload(slimTSF)
from slimTSF import (
    build_dataset_from_slices,
    build_feature_vector,
    discover_slice_files,
    rank_features_via_rf,
 )

DATA_ROOT = Path('/Users/ran/cs/6850/project/data/2026-05-08')
GSEP_INDEX_CSV = Path('/Users/ran/cs/6850/project/data/2026-05-08-train-indexafter1990.csv')
EVENTS_JSON = Path('/Users/ran/cs/6850/project/data/2026-04-29/events/merge/all_events_merged.json')
OUTPUT_DIR = Path('/Users/ran/cs/6850/project/data/training_results/leakage_free')

VALUE_COLS = ['p3_flux_ic']
WINDOW_SIZE = 12
STEP_SIZE = 1
POOLING_WIDTH = 3
BACKGROUND_WINDOW_POINTS = None
BACKGROUND_SIGMA_CLIP = 3.0
ONLY_RELATIVE_FEATURES = True

RANDOM_STATE = 42
N_SPLITS = 5
N_RANK_EXPERIMENTS = 8

In [2]:
def _slice_key_from_name(name: str):
    stem = Path(name).stem
    match = re.search(r'(\d{4}-\d{2}-\d{2})[T_ ]?(\d{2})-(\d{2})(?:-(\d{2}))?', stem)
    if match:
        return f"{match.group(1)}_{match.group(2)}-{match.group(3)}"
    return None

with EVENTS_JSON.open('r') as fh:
    events = json.load(fh)

event_id_by_slice_key = {}
for event in events:
    slice_paths = event.get('slice_path')
    if not slice_paths:
        continue
    if isinstance(slice_paths, str):
        slice_paths = [slice_paths]
    for slice_path in slice_paths:
        slice_key = _slice_key_from_name(slice_path)
        if slice_key and slice_key not in event_id_by_slice_key:
            event_id_by_slice_key[slice_key] = f"event-{slice_key}"

index_frame = pd.read_csv(GSEP_INDEX_CSV)
label_by_slice_key = {}
group_key_by_slice_key = {}
for _, row in index_frame.iterrows():
    file_root = str(row.get('file_root', '')).strip()
    identifier = str(row.get('identifier', '')).strip()
    label = int(row.get('label', 0))
    slice_key = _slice_key_from_name(file_root)
    if not slice_key:
        continue
    label_by_slice_key[slice_key] = label
    if identifier:
        event_id_by_slice_key[slice_key] = identifier
        group_key_by_slice_key[slice_key] = identifier.split('_')[0]
    else:
        group_key_by_slice_key[slice_key] = None

gsep_index = {key: {'label': '1'} for key, label in label_by_slice_key.items() if label == 1}

print(f'Event map size: {len(event_id_by_slice_key)}')
print(f'Labeled slices: {len(label_by_slice_key)}')
print(f'Positive slices: {len(gsep_index)}')

Event map size: 1475
Labeled slices: 1475
Positive slices: 841


In [3]:
slice_paths = discover_slice_files(DATA_ROOT)

def _is_usable_slice(path, required_cols):
    try:
        df_head = pd.read_csv(path, nrows=50)
    except Exception:
        return False
    if 'time_tag' not in df_head.columns:
        return False
    for col in required_cols:
        if col not in df_head.columns:
            return False
        series = pd.to_numeric(df_head[col], errors='coerce')
        if not np.isfinite(series.to_numpy(dtype=float)).any():
            return False
    return True

usable_paths = [p for p in slice_paths if _is_usable_slice(p, VALUE_COLS)]
skipped = len(slice_paths) - len(usable_paths)
print(f'Usable slices: {len(usable_paths)} | Skipped (missing/empty {VALUE_COLS}): {skipped}')

X, y, metadata = build_dataset_from_slices(
    usable_paths,
    gsep_index=gsep_index,
    time_col='time_tag',
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
    pooling_width=POOLING_WIDTH,
    value_cols=VALUE_COLS,
    background_window_points=BACKGROUND_WINDOW_POINTS,
    background_sigma_clip=BACKGROUND_SIGMA_CLIP,
    only_relative_features=ONLY_RELATIVE_FEATURES,
 )

event_hits = 0
background_hits = 0
for meta in metadata:
    group_key = group_key_by_slice_key.get(meta.slice_key)
    if group_key:
        meta.group_key = group_key
        event_hits += 1
    else:
        meta.group_key = f'background-{meta.slice_key[:10]}'
        background_hits += 1

groups = np.asarray([m.group_key for m in metadata], dtype=str)
slice_dates = pd.to_datetime([m.slice_key for m in metadata], errors='coerce')

print(f'Total slices: {len(metadata)}')
print(f'Event-mapped slices: {event_hits}')
print(f'Background slices: {background_hits}')
print(f'Class balance: {dict(zip(*np.unique(y, return_counts=True)))}')
print(f'Unique groups: {len(set(groups))}')

Usable slices: 1792 | Skipped (missing/empty ['p3_flux_ic']): 232
Total slices: 1792
Event-mapped slices: 1792
Background slices: 0
Class balance: {np.int64(0): np.int64(992), np.int64(1): np.int64(800)}
Unique groups: 984


/var/folders/tk/bt_xb0t50438rf881ft5bzg80000gn/T/ipykernel_50223/1912006653.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  slice_dates = pd.to_datetime([m.slice_key for m in metadata], errors='coerce')


In [4]:
def _tss(tp, tn, fp, fn):
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    return float(sensitivity + specificity - 1.0)

def _hss(tp, tn, fp, fn):
    numerator = 2.0 * (tp * tn - fn * fp)
    denominator = (tp + fn) * (fn + tn) + (tp + fp) * (fp + tn)
    if denominator == 0:
        return 0.0
    return float(numerator / denominator)

def _positive_proba(model, X, positive_label=1):
    if len(getattr(model, 'classes_', [])) < 2:
        return np.zeros(len(X), dtype=float)
    probs = model.predict_proba(X)
    if positive_label in model.classes_:
        pos_idx = int(np.where(model.classes_ == positive_label)[0][0])
        return probs[:, pos_idx]
    return probs[:, -1]

def _scores_from_probs(y_true, probs, threshold):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    return {
        'tp': int(tp),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tss': _tss(tp, tn, fp, fn),
        'hss': _hss(tp, tn, fp, fn),
        'f1': f1_score(y_true, preds, zero_division=0),
    }

def _best_threshold(y_true, probs):
    thresholds = np.linspace(0.05, 0.95, 19)
    best = { 'threshold': 0.5, 'tss': -np.inf, 'hss': -np.inf }
    for threshold in thresholds:
        scores = _scores_from_probs(y_true, probs, threshold)
        if scores['tss'] > best['tss'] or (math.isclose(scores['tss'], best['tss']) and scores['hss'] > best['hss']):
            best = { 'threshold': float(threshold), 'tss': scores['tss'], 'hss': scores['hss'] }
    return best['threshold'], best

gkf = GroupKFold(n_splits=min(N_SPLITS, len(np.unique(groups))))

fold_results = []
all_val_probs = []
all_val_true = []

for fold_index, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups), start=1):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    rank_summary = rank_features_via_rf(
        X_train,
        y_train,
        n_experiments=N_RANK_EXPERIMENTS,
        random_state=RANDOM_STATE,
    )
    selected = rank_summary.selected_features

    model = RandomForestClassifier(
        n_estimators=400,
        random_state=RANDOM_STATE + fold_index,
        class_weight='balanced',
        n_jobs=-1,
    )
    model.fit(X_train[selected], y_train)
    probs = _positive_proba(model, X_val[selected])

    all_val_probs.extend(probs.tolist())
    all_val_true.extend(y_val.tolist())

    fold_threshold, _ = _best_threshold(y_val.to_numpy(), probs)
    fold_scores = _scores_from_probs(y_val.to_numpy(), probs, fold_threshold)
    fold_scores.update({ 'fold': fold_index, 'threshold': fold_threshold })
    fold_results.append(fold_scores)

cv_threshold, cv_best = _best_threshold(np.asarray(all_val_true), np.asarray(all_val_probs))
cv_scores = _scores_from_probs(np.asarray(all_val_true), np.asarray(all_val_probs), cv_threshold)

print('--- GroupKFold Results (Event-ID Grouping) ---')
for result in fold_results:
    print(f"Fold {result['fold']}: TSS={result['tss']:.3f}, HSS={result['hss']:.3f}, F1={result['f1']:.3f}, thr={result['threshold']:.2f}")

print('\n--- CV Aggregate (All Validation Slices) ---')
print(f"Best Threshold: {cv_threshold:.2f}")
print(f"TSS={cv_scores['tss']:.3f}, HSS={cv_scores['hss']:.3f}, F1={cv_scores['f1']:.3f}")

--- GroupKFold Results (Event-ID Grouping) ---
Fold 1: TSS=0.917, HSS=0.911, F1=0.952, thr=0.30
Fold 2: TSS=0.954, HSS=0.950, F1=0.973, thr=0.35
Fold 3: TSS=0.919, HSS=0.910, F1=0.951, thr=0.10
Fold 4: TSS=0.925, HSS=0.921, F1=0.956, thr=0.55
Fold 5: TSS=0.920, HSS=0.916, F1=0.955, thr=0.35

--- CV Aggregate (All Validation Slices) ---
Best Threshold: 0.35
TSS=0.915, HSS=0.909, F1=0.951


In [5]:
meta_frame = pd.DataFrame({
    'group': groups,
    'slice_key': [m.slice_key for m in metadata],
    'label': y,
})
meta_frame['slice_date'] = pd.to_datetime(meta_frame['slice_key'], errors='coerce')

group_dates = meta_frame.groupby('group')['slice_date'].min().sort_values()
cut_index = max(1, int(len(group_dates) * 0.8))
train_groups = set(group_dates.index[:cut_index])
test_groups = set(group_dates.index[cut_index:])

train_idx = [i for i, group in enumerate(groups) if group in train_groups]
test_idx = [i for i, group in enumerate(groups) if group in test_groups]

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]
X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]

rank_summary = rank_features_via_rf(
    X_train,
    y_train,
    n_experiments=N_RANK_EXPERIMENTS,
    random_state=RANDOM_STATE,
 )
selected = rank_summary.selected_features

model = RandomForestClassifier(
    n_estimators=400,
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_jobs=-1,
 )
model.fit(X_train[selected], y_train)
test_probs = _positive_proba(model, X_test[selected])
test_scores = _scores_from_probs(y_test.to_numpy(), test_probs, cv_threshold)

print('--- Temporal Holdout ---')
print(f'Train groups: {len(train_groups)} | Test groups: {len(test_groups)}')
print(f"TSS={test_scores['tss']:.3f}, HSS={test_scores['hss']:.3f}, F1={test_scores['f1']:.3f}")

/var/folders/tk/bt_xb0t50438rf881ft5bzg80000gn/T/ipykernel_50223/2401622345.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  meta_frame['slice_date'] = pd.to_datetime(meta_frame['slice_key'], errors='coerce')


--- Temporal Holdout ---
Train groups: 787 | Test groups: 197
TSS=-0.023, HSS=0.000, F1=0.000


In [6]:
X_logged = np.log10(X.clip(lower=1e-6))

final_rank = rank_features_via_rf(
    X_logged,
    y,
    n_experiments=N_RANK_EXPERIMENTS,
    random_state=RANDOM_STATE,
    top_k=15
 )
final_features = final_rank.selected_features

final_model = RandomForestClassifier(
    n_estimators=500,
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_jobs=-1,
 )
final_model.fit(X[final_features], y)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model_path = OUTPUT_DIR / 'slimtsf_rf_leakage_free.joblib'
joblib.dump(final_model, model_path)

threshold_path = OUTPUT_DIR / 'slimtsf_threshold.json'
threshold_path.write_text(json.dumps({
    'threshold': float(cv_threshold),
    'tss': float(cv_scores['tss']),
    'hss': float(cv_scores['hss']),
    'f1': float(cv_scores['f1']),
}, indent=2))

features_path = OUTPUT_DIR / 'slimtsf_selected_features.json'
features_path.write_text(json.dumps({
    'selected_features': final_features,
    'feature_count': len(final_features),
}, indent=2))

print(f'Model saved to: {model_path}')
print(f'Threshold saved to: {threshold_path}')
print(f'Selected features saved to: {features_path}')

Model saved to: /Users/ran/cs/6850/project/data/training_results/leakage_free/slimtsf_rf_leakage_free.joblib
Threshold saved to: /Users/ran/cs/6850/project/data/training_results/leakage_free/slimtsf_threshold.json
Selected features saved to: /Users/ran/cs/6850/project/data/training_results/leakage_free/slimtsf_selected_features.json


In [7]:
print(metadata[0].group_key, metadata[1].group_key)

event-1978-09-23 event-1978-09-23


In [10]:
results = []

print(f"Processing {len(slice_paths)} files...")

for path in slice_paths:
    try:
        # 1. Read and Log-Transform
        df = pd.read_csv(path)
        df['p3_flux_ic'] = np.log10(df['p3_flux_ic'].clip(lower=1e-6))
        
        df['time_tag'] = pd.to_datetime(df['time_tag'], errors='coerce')
        df = df.dropna(subset=['time_tag']).sort_values('time_tag')
        
        if df.empty:
            continue

        # 2. Extract features
        vector, _ = build_feature_vector(
            df,
            time_col='time_tag',
            window_size=WINDOW_SIZE,
            step_size=STEP_SIZE,
            pooling_width=POOLING_WIDTH,
            value_cols=VALUE_COLS,
            background_window_points=BACKGROUND_WINDOW_POINTS,
            background_sigma_clip=BACKGROUND_SIGMA_CLIP,
            only_relative_features=ONLY_RELATIVE_FEATURES,
        )
        
        if len(vector) != len(X.columns):
            continue
            
        vector_df = pd.DataFrame([vector], columns=X.columns)
        
        # 3. Predict
        prob = _positive_proba(final_model, vector_df[final_features])[0]

        # 4. Collection logic (Saving features + Metadata)
        if prob >= 0.10: 
            # Get the actual feature values used for this prediction
            feature_dict = vector_df[final_features].iloc[0].to_dict()
            
            # Identify the unique start time for THIS specific file
            current_start_time = df['time_tag'].iloc[0]
            
            entry = {
                'file_name': path.name,
                'file_root': str(path.absolute().parent),
                'start_time': current_start_time,
                'probability': round(float(prob), 4),
                'status': 'EVENT_DETECTED' if prob >= 0.35 else 'LOW_CONFIDENCE'
            }
            
            # Merge the feature values into the dictionary
            entry.update(feature_dict)
            results.append(entry)
            
    except Exception as e:
        # print(f"Error on {path.name}: {e}")
        continue

# 5. Save and Display
results_df = pd.DataFrame(results)

if not results_df.empty:
    output_path = "model_detailed_reportafter1990.csv"
    results_df.to_csv(output_path, index=False)
    
    print(f"\n--- SUCCESS: {len(results_df)} detections saved to {output_path} ---")
    
    # Show the top 5 detections with their paths and top features
    view_cols = ['file_name', 'start_time', 'probability', 'status'] + final_features[:2]
    print(results_df.sort_values('probability', ascending=False)[view_cols].head(5))
else:
    print("No events met the threshold.")

Processing 2024 files...

--- SUCCESS: 199 detections saved to model_detailed_reportafter1990.csv ---
                   file_name          start_time  probability          status  \
102  1975-08-20T15-35-00.csv 1975-08-20 15:35:00        1.000  EVENT_DETECTED   
103  1975-08-20T23-40-00.csv 1975-08-20 23:40:00        0.998  EVENT_DETECTED   
121  1975-09-11T08-30-00.csv 1975-09-11 08:30:00        0.998  EVENT_DETECTED   
120  1975-09-10T06-50-00.csv 1975-09-10 06:50:00        0.996  EVENT_DETECTED   
78   1975-07-17T14-30-00.csv 1975-07-17 14:30:00        0.996  EVENT_DETECTED   

     p3_flux_ic__global_std  p3_flux_ic__std__pool_max  
102                1.438829                   1.397864  
103                1.579985                   1.445886  
121                1.905347                   1.667453  
120                1.775890                   1.569738  
78                 3.204827                   3.201734  


In [9]:
true_event_starts = pd.Series(true_event_starts)
results = []
for path in slice_paths:
    try:
        # 1. Read once and transform immediately
        df = pd.read_csv(path)
        df['p3_flux_ic'] = np.log10(df['p3_flux_ic'].clip(lower=1e-6))
        
        df['time_tag'] = pd.to_datetime(df['time_tag'], errors='coerce')
        df = df.dropna(subset=['time_tag']).sort_values('time_tag')
        
        if df.empty:
            continue

        # 2. Extract features from the LOGGED data
        vector, _ = build_feature_vector(
            df,
            time_col='time_tag',
            window_size=WINDOW_SIZE,
            step_size=STEP_SIZE,
            pooling_width=POOLING_WIDTH,
            value_cols=VALUE_COLS,
            background_window_points=BACKGROUND_WINDOW_POINTS,
            background_sigma_clip=BACKGROUND_SIGMA_CLIP,
            only_relative_features=ONLY_RELATIVE_FEATURES,
        )
        
        if len(vector) != len(X.columns):
            continue
            
        vector_df = pd.DataFrame([vector], columns=X.columns)
        prob = _positive_proba(final_model, vector_df[final_features])[0]
        print(prob)

        if prob >= 0.10: # If probability is high (like your 0.936)
            slice_start = df['time_tag'].iloc[0]
            # Calculate if this hit is near a real event
            time_diffs = np.abs((true_event_starts - slice_start).dt.total_seconds() / 60)
            is_correct = np.any(time_diffs <= 30)

        results.append({
            'file': path.name,
            'start_time': slice_start,
            'probability': float(prob),
            'status': 'GOOD PREDICTION' if is_correct else 'FALSE ALARM',
        })
        print(f"File: {path.name} | Prob: {prob:.3f}")
            
    except Exception as e:
        continue

# 4. Final Display
results_df = pd.DataFrame(results)
if not results_df.empty:
    output_path = "model_detection_report.csv"
    results_df.to_csv(output_path, index=False)
    
    print(f"\n--- SUCCESS: {len(results_df)} detections saved to {output_path} ---")
    print("\nTop Detections with Features:")
    # Displaying metadata and probability (hiding full feature list for brevity in print)
    display_cols = ['file_name', 'probability', 'prediction_result'] + [final_features[0], final_features[1]]
    print(results_df.sort_values('probability', ascending=False)[display_cols].head(10))
else:
    print("No events met the threshold. CSV not created.")

NameError: name 'true_event_starts' is not defined